# FDSI benchmark v1.1: optimisation -- pareto (wh_loss/sv_loss)

The v1.1 counterpart of `../fdsi_benchmark/03a_pareto_optimisation.ipynb`: one multi-objective
search over `(wh_loss, sv_loss)`, whose front is then read twice -- once by `min_sv_loss` (v1.0's
rule) and once by `knee` (new in v1.1) -- giving the two Pareto branches `04a` applies.

## Config

In [8]:
RUN_OPTIMISATION = False  # run the search when no complete cached study exists

import yaml
from pathlib import Path

with open('../../configs/data_configs/fdsi_benchmark_grid.yaml') as f:
    grid = yaml.safe_load(f)

DATA_DIR = Path(grid['data_root'])           # Raw archive root: <sub>/{clean,noisy}/
OUTPUTS_ROOT = Path(grid['outputs_root'])    # Precomputed-outputs archive root
CAL_DIR = OUTPUTS_ROOT / 'calibration'       # Shared with v1.0 -- written by ../fdsi_benchmark/01_calibration
ADAPT_DIR = OUTPUTS_ROOT / 'adaptation_v1_1' # v1.1 results
ADAPT_DIR_V10 = OUTPUTS_ROOT / 'adaptation'  # Cached v1.0 results, read-only here
ADAPT_DIRS = {'v1.0': ADAPT_DIR_V10, 'v1.1': ADAPT_DIR}
OPT_DIR = ADAPT_DIR / 'optimisation'

FS, EXT_FACT = grid['fs'], grid['ext_fact']
TOL_SPIKE_MS, ROA_CAL_TH = grid['tol_spike_ms'], grid['roa_cal_th']
CAL_END, ISO_DUR = int(grid['cal_duration_s'] * FS), int(grid['iso_duration_s'] * FS)

SUBJECTS, CONDITIONS, SNR_LEVELS = grid['subjects'], grid['conditions'], grid['snr_levels']
TRIANGULAR = [c for c in CONDITIONS if 'triangular' in c]

POOL_CONDITIONS = grid['pool_conditions']
HOLDOUT_CONDITIONS = [c for c in CONDITIONS if c not in POOL_CONDITIONS]
OPTIM_SUB, OPTIM_SNR, N_TRIALS = grid['optim_sub'], grid['optim_snr'], grid['n_trials']

BASE_CONFIG_PATH = Path('../../configs/adapt_configs/default_muniverse.yaml')  # v1.0's base config
LR_MODE = 'fixed'   # v1.1 searches lr_fixed only -- the branch v1.0 promoted as its headline
DEVICE = 'cpu'      # deterministic, and MPS lacks the QR the sv update needs

OBJECTIVES = ('wh_loss', 'sv_loss')

print(f'Apply-to-all grid: {len(SUBJECTS)} subjects x {len(CONDITIONS)} conditions x {len(SNR_LEVELS)} SNR levels '
      f'= {len(SUBJECTS) * len(CONDITIONS) * len(SNR_LEVELS)} recordings')
print(f'Calibration window [0, {CAL_END}) samples; adaptation starts at sample {CAL_END}')

Apply-to-all grid: 5 subjects x 5 conditions x 4 SNR levels = 100 recordings
Calibration window [0, 10240) samples; adaptation starts at sample 10240


## Data pool

The three pooled recordings `load_data` returns are the whole recordings; v1.1 searches the
segment after the calibration window, so each one is trimmed to `[CAL_END, end)` here. With
`source_fifo_from_calib=True` in the base config below, every trial then adapts forwards from
the calibration's end and is scored only on signal it was never fitted on.

One approximation: `emg[CAL_END:]` is filtered on its own, so the filters restart at `CAL_END`
(a few-ms transient), unlike `process_from_calib_end` in `02`/`04*`.

In [9]:
import pickle
import shutil
import sys
import time
from dataclasses import replace
from pathlib import Path
from typing import Dict
sys.path.insert(0, '../..')
sys.path.insert(0, '../fdsi_benchmark')  # for fdsi_common
sys.path.insert(0, str(Path.cwd()))

import optuna
from adapt_decomp import AdaptationResult, load_data
from adapt_decomp.adaptation import AdaptConfig
from adapt_decomp.adaptation.optimize import DEFAULT_PARAM_SPACE, optimize_adapt_decomp
import fdsi_common as fc
import fdsi_v11 as fv

DATA_CONFIG_PATH = Path('../../configs/data_configs/fdsi_pool_memory_example.yaml')
with DATA_CONFIG_PATH.open() as f:
    data_config = yaml.safe_load(f)

pool = load_data(data_config)
assert set(pool) == set(POOL_CONDITIONS), pool.keys()

# Restrict every pooled recording to the samples after its calibration window
pool = {name: replace(dataset,
                      emg=dataset.emg[CAL_END:],
                      gt_paired_bin=None if dataset.gt_paired_bin is None
                                    else dataset.gt_paired_bin[CAL_END:])
        for name, dataset in pool.items()}

print(f'Search space: {DEFAULT_PARAM_SPACE}')
{name: tuple(dataset.emg.shape) for name, dataset in pool.items()}

Search space: {'wh_learning_rate': ('log_float', 0.0001, 0.05), 'sv_learning_rate': ('log_float', 0.0001, 0.1), 'centroid_momentum': ('float', 0.0, 0.95)}


{'triangular-ramp40s': (174080, 100),
 'triangular-ramp10s': (174080, 100),
 'triangular-ramp5s': (174080, 100)}

## Optimisation

In [10]:
# v1.0's base config, set up for a v1.1 search
base_config = AdaptConfig.from_yaml(BASE_CONFIG_PATH)
base_config.lr_mode = LR_MODE
base_config.source_fifo_from_calib = True   # seed the source FIFO from the calibration's tail
base_config.sv_loss_reduction = 'mean'      # per-unit mean, so the loss doesn't scale with unit count
base_config.device = DEVICE

# One search setting for every v1.1 search: one-at-a-time trials (n_jobs=1, reproducible),
# all available cores (n_cores=None), v1.0's seed, and CoV-ISI unit selection.
SEARCH = dict(n_jobs=1, n_cores=None, random_seed=42,
              unit_selection='unsupervised', unit_selection_kwargs={'cov_th': 0.3})

print(f'base: lr_mode={base_config.lr_mode}  sv_loss_reduction={base_config.sv_loss_reduction}  '
      f'source_fifo_from_calib={base_config.source_fifo_from_calib}')
print(f'search: {SEARCH}')

base: lr_mode=fixed  sv_loss_reduction=mean  source_fifo_from_calib=True
search: {'n_jobs': 1, 'n_workers': 3, 'random_seed': 42, 'unit_selection': 'unsupervised', 'unit_selection_kwargs': {'cov_th': 0.3}}


In [11]:
SEARCH_DIR = fv.SEARCH_DIRS['pareto']
opt_dir = OPT_DIR / fc.LR_MODE_TOKEN[LR_MODE] / SEARCH_DIR
study_path = opt_dir / 'study.pkl'

study = None
if study_path.exists():
    with open(study_path, 'rb') as f:
        study = pickle.load(f)

n_complete = 0 if study is None else sum(t.state.name == 'COMPLETE' for t in study.trials)

if n_complete >= N_TRIALS:
    print(f'{SEARCH_DIR}: loading cached study ({n_complete} complete trials).')
    pareto_front = study.best_trials if True else None
elif RUN_OPTIMISATION:
    shutil.rmtree(opt_dir, ignore_errors=True)
    print(f'Running {SEARCH_DIR} ({N_TRIALS} trials, objectives=OBJECTIVES) ...')
    start = time.perf_counter()
    result = optimize_adapt_decomp(
        pool=pool,
        objectives=OBJECTIVES,
        param_space=DEFAULT_PARAM_SPACE,
        base_config=base_config,
        selection='min_sv_loss',
        n_trials=N_TRIALS,
        compute_roa=True,
        roa_kwargs={'tol_spike_ms': TOL_SPIKE_MS},
        best_result_path=str(opt_dir),
        **SEARCH,
    )
    study, pareto_front = result.study, result.pareto_front
    study.set_user_attr('wall_time_s', time.perf_counter() - start)
    with open(opt_dir / 'study.pkl', 'wb') as f:
        pickle.dump(study, f)
else:
    raise FileNotFoundError(f'No complete cached study in {opt_dir} and RUN_OPTIMISATION=False.')

print(f'  completed trials: {sum(t.state.name == "COMPLETE" for t in study.trials)}')
print(f'  wall time: {study.user_attrs.get("wall_time_s", float("nan")):.0f} s')

mtpe_pareto: loading cached study (50 complete trials).
  completed trials: 50
  wall time: 3790 s


### Optimisation landscape -- RoA vs pooled wh_loss/sv_loss/total_loss

In [35]:
from pprint import pprint
pprint(selected['knee'].user_attrs['roa_mean_pooled'])
pprint(selected['min_sv_loss'].user_attrs['roa_mean_pooled'])


55.02980939597609
58.17876269271025


In [46]:
from adapt_decomp.utils.plots import plot_optimisation_landscape_grid, plot_pareto_scatter_grid
from adapt_decomp.adaptation.optimize import SELECTION_RULES

selected = {name: SELECTION_RULES[name](pareto_front) for name in ('min_sv_loss', 'knee')}
for name, trial in selected.items():
    print(
        f"{name:12} -> trial {trial.number}"
        f" values = {[round(v, 4) for v in trial.values]},"
        f" mean_roa = {trial.user_attrs['roa_mean_pooled']}"
    )


# The _grid variants return a go.Figure; the singular plot_optimisation_landscape /
# plot_pareto_scatter return bare traces, for composing into a figure yourself.
fig = plot_optimisation_landscape_grid({'pareto (v1.1)': study},
                                       {'pareto (v1.1)': 'pareto (v1.1)'},
                                       {'pareto (v1.1)': selected['min_sv_loss']})
fig.show()

min_sv_loss  -> trial 46 values = [342722.9453, 20.9783], mean_roa = 58.17876269271025
knee         -> trial 48 values = [5293.0697, 22.5974], mean_roa = 55.02980939597609


### Pareto front -- wh_loss vs sv_loss, coloured by RoA

In [13]:
fig = plot_pareto_scatter_grid(
    {'min-sv selection': study, 'knee selection': study},
    {'min-sv selection': pareto_front, 'knee selection': pareto_front},
    {'min-sv selection': selected['min_sv_loss'], 'knee selection': selected['knee']},
)
fig.show()

## Promote both front selections

Each selection's resolved config was already written by the search as
`trial_<n>/config.yaml`, so promoting is a read and a re-save under the branch's own name.

In [14]:
for branch, name in (('pareto_min_sv', 'min_sv_loss'), ('pareto_knee', 'knee')):
    trial = selected[name]
    best_config = AdaptConfig.from_yaml(opt_dir / f'trial_{trial.number}' / 'config.yaml')
    path = fv.promoted_path(branch)
    best_config.to_yaml(path)
    print(f'{branch:14} trial {trial.number:3}  wh_lr={best_config.wh_learning_rate:.4g}  '
          f'sv_lr={best_config.sv_learning_rate:.4g}  '
          f'centroid_momentum={best_config.centroid_momentum:.3g}  -> {path.name}')

pareto_min_sv  trial  46  wh_lr=0.03956  sv_lr=0.001667  centroid_momentum=0.737  -> optim_muniverse_fdsi_v11_pareto_min_sv.yaml
pareto_knee    trial  48  wh_lr=0.001165  sv_lr=0.0052  centroid_momentum=0.921  -> optim_muniverse_fdsi_v11_pareto_knee.yaml


### Reading this

`min_sv_loss` takes the front's lowest-`sv_loss` member, which is what v1.0 promoted; `knee`
takes the point of maximum curvature, trading a little `sv_loss` for a large `wh_loss` gain.
`04a` applies both to all 100 recordings so the two selections can be compared directly.